# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alsa-mirza/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

How to use the ranked queue:
The queue is designed as a decision support tool instead of an automation for content production. Items are prioritized according to observed signals and labeled according to a set of rules to better explain the rationale for human action.

Position Declining - Review and Refresh: The item’s observed average position has declined since the previous period. We recommend reviewing the content and considering a refresh if the position has become worse.

CTR Declining - Review Metadata: The item’s observed recent CTR is below the previous period’s CTR. We recommend reviewing the metadata/content presentation before taking other action.

Monitor: The signals were insufficient to identify an item requiring intervention. Items in this set should be monitored but not changed.

The thresholds and labels represent directional decision support rules applied to the observed data distribution. They represent recommendations for humans to evaluate the situation before making any changes to content.

In [14]:
from google.colab import userdata
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_TOKEN")
print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [15]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    split="train",
    token=HF_TOKEN
)
df = dataset.to_pandas()
print("Data loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Data loaded successfully!
Shape: (2414248, 21)

Columns:
['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


In [16]:
# Week 7: inspect the fields available for the action playbook
print("Total rows:", len(df))
print("\nAll columns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

Total rows: 2414248

All columns:
1. client_hash_id
2. content_hash_id
3. query_hash_id
4. query_char_count
5. query_token_count
6. window_start
7. window_end
8. impressions_90d
9. clicks_90d
10. impressions_last30
11. clicks_last30
12. impressions_prev30
13. clicks_prev30
14. avg_position_90d
15. avg_position_last30
16. avg_position_prev30
17. content_total_impressions_90d
18. content_visible_query_count
19. rare_query_count
20. rare_impressions_share
21. anonymized_impressions_share


In [17]:
# Week 7 - Build practical content signals
import numpy as np
import pandas as pd
# Avoid division by zero
df["ctr_90d"] = (
    df["clicks_90d"] /
    df["impressions_90d"].replace(0, np.nan)
)
df["ctr_last30"] = (
    df["clicks_last30"] /
    df["impressions_last30"].replace(0, np.nan)
)
df["ctr_prev30"] = (
    df["clicks_prev30"] /
    df["impressions_prev30"].replace(0, np.nan)
)
# Change in CTR
df["ctr_change"] = df["ctr_last30"] - df["ctr_prev30"]
# Change in average position
# Lower position number generally means a better ranking position.
df["position_change"] = (
    df["avg_position_last30"] -
    df["avg_position_prev30"]
)
print("Signals created successfully!")
print("\nNew signal columns:")
print([
    "ctr_90d",
    "ctr_last30",
    "ctr_prev30",
    "ctr_change",
    "position_change"
])
print("\nSignal preview:")
display(
    df[
        [
            "ctr_90d",
            "ctr_last30",
            "ctr_prev30",
            "ctr_change",
            "position_change"
        ]
    ].head()
)

Signals created successfully!

New signal columns:
['ctr_90d', 'ctr_last30', 'ctr_prev30', 'ctr_change', 'position_change']

Signal preview:


,ctr_90d,ctr_last30,ctr_prev30,ctr_change,position_change
0,0.0,NaN,0.0,NaN,NaN
1,0.0,NaN,0.0,NaN,NaN
2,0.0,0.0,0.0,0.0,2.272727
3,0.0,0.0,0.0,0.0,13.000000
4,0.0,NaN,NaN,NaN,NaN


In [18]:
# Week 7 - Inspect signal distributions before defining actions
signal_cols = [
    "ctr_90d",
    "ctr_last30",
    "ctr_prev30",
    "ctr_change",
    "position_change"
]
print("Signal summary:")
display(df[signal_cols].describe().T)

print("\nMissing values:")
display(df[signal_cols].isna().sum())

print("\nSelected percentiles:")
display(
    df[signal_cols].quantile(
        [0.10, 0.25, 0.50, 0.75, 0.90]
    ).T
)

Signal summary:


,count,mean,std,min,25%,50%,75%,max
ctr_90d,2414248.0,0.002031,0.010800,0.000000,0.000000,0.0,0.0,0.633333
ctr_last30,1883490.0,0.002614,0.026826,0.000000,0.000000,0.0,0.0,1.000000
ctr_prev30,2084828.0,0.002643,0.025973,0.000000,0.000000,0.0,0.0,1.000000
ctr_change,1704767.0,-0.000201,0.036404,-1.000000,0.000000,0.0,0.0,1.000000
position_change,1704767.0,1.487498,15.360744,-568.555556,-2.389043,0.2,4.0,581.500000



Missing values:


,0
ctr_90d,0
ctr_last30,530758
ctr_prev30,329420
ctr_change,709481
position_change,709481



Selected percentiles:


,0.10,0.25,0.50,0.75,0.90
ctr_90d,0.000000,0.000000,0.0,0.0,0.0
ctr_last30,0.000000,0.000000,0.0,0.0,0.0
ctr_prev30,0.000000,0.000000,0.0,0.0,0.0
ctr_change,0.000000,0.000000,0.0,0.0,0.0
position_change,-9.991194,-2.389043,0.2,4.0,15.0


In [19]:
# Week 7 - Ranked actions + reason codes

# Work on a copy so the original dataframe remains available
playbook = df.copy()

# Start with a neutral default
playbook["action"] = "Monitor"
playbook["reason_code"] = "NO_CLEAR_SIGNAL"
playbook["priority_score"] = 0.0

# ---------------------------------------------------------
# Rule 1: Refresh candidate
# Recent position is meaningfully worse than previous period.
# Higher position_change = movement toward a worse position.
# ---------------------------------------------------------
refresh_mask = (
    playbook["position_change"].notna()
    & (playbook["position_change"] >= 15)
)

playbook.loc[refresh_mask, "action"] = "Review and Refresh"
playbook.loc[refresh_mask, "reason_code"] = "POSITION_DECLINE"
playbook.loc[refresh_mask, "priority_score"] = (
    playbook.loc[refresh_mask, "position_change"]
)

# ---------------------------------------------------------
# Rule 2: CTR decline
# Only use this when both CTR periods are actually available.
# ---------------------------------------------------------
ctr_decline_mask = (
    playbook["ctr_change"].notna()
    & (playbook["ctr_change"] < 0)
    & ~refresh_mask
)

playbook.loc[ctr_decline_mask, "action"] = "Review Metadata"
playbook.loc[ctr_decline_mask, "reason_code"] = "CTR_DECLINE"
playbook.loc[ctr_decline_mask, "priority_score"] = (
    -playbook.loc[ctr_decline_mask, "ctr_change"] * 100
)

# ---------------------------------------------------------
# Rule 3: Monitor everything without a strong signal
# ---------------------------------------------------------

# Rank highest-priority actions first
playbook = playbook.sort_values(
    "priority_score",
    ascending=False
).reset_index(drop=True)

# Add a rank
playbook["action_rank"] = range(1, len(playbook) + 1)

print("Action playbook created successfully!")

print("\nAction counts:")
display(playbook["action"].value_counts())

print("\nReason-code counts:")
display(playbook["reason_code"].value_counts())

print("\nTop 10 actions:")
display(
    playbook[
        [
            "action_rank",
            "client_hash_id",
            "content_hash_id",
            "query_hash_id",
            "action",
            "reason_code",
            "priority_score"
        ]
    ].head(10)
)

Action playbook created successfully!

Action counts:


,count
action,
Monitor,2168368
Review and Refresh,170752
Review Metadata,75128



Reason-code counts:


,count
reason_code,
NO_CLEAR_SIGNAL,2168368
POSITION_DECLINE,170752
CTR_DECLINE,75128



Top 10 actions:


,action_rank,client_hash_id,content_hash_id,query_hash_id,action,reason_code,priority_score
0,1,client_23a62021009f63c4,content_884399c4f70bbedc,query_23797ac3cf508604,Review and Refresh,POSITION_DECLINE,581.500000
1,2,client_20259bd6705d81d4,content_f90fd9a7a7ef5bc3,query_3115c08d1c6cb1f8,Review and Refresh,POSITION_DECLINE,552.600000
2,3,client_20259bd6705d81d4,content_6cffe9e76a03d4e4,query_d52187abb4d40985,Review and Refresh,POSITION_DECLINE,505.055556
3,4,client_20259bd6705d81d4,content_ab294a2f95286b64,query_be305f1213d9af03,Review and Refresh,POSITION_DECLINE,433.000000
4,5,client_23a62021009f63c4,content_cd32ab23b32de847,query_f6dfa923811062c4,Review and Refresh,POSITION_DECLINE,409.000000
5,6,client_23a62021009f63c4,content_41932abf133a03d8,query_c137be6212b629e6,Review and Refresh,POSITION_DECLINE,408.500000
6,7,client_23a62021009f63c4,content_6df60b5a01b70af2,query_a7935a036f0a65c4,Review and Refresh,POSITION_DECLINE,402.300000
7,8,client_23a62021009f63c4,content_fd88ea449da84b00,query_858656074add0998,Review and Refresh,POSITION_DECLINE,393.000000
8,9,client_20259bd6705d81d4,content_389c468924980ec4,query_9d1101807697545b,Review and Refresh,POSITION_DECLINE,391.000000
9,10,client_20259bd6705d81d4,content_e308834a4d3d2143,query_eb4f9229aa32e59b,Review and Refresh,POSITION_DECLINE,374.166667


In [20]:
# Week 7 - Archetype to action mapping
archetype_map = {
    "POSITION_DECLINE": {
        "archetype": "Position Declining",
        "action": "Review and Refresh",
        "human_reason": "Recent average position moved materially worse than the previous period."
    },
    "CTR_DECLINE": {
        "archetype": "CTR Declining",
        "action": "Review Metadata",
        "human_reason": "Recent CTR was lower than the previous observed period."
    },
    "NO_CLEAR_SIGNAL": {
        "archetype": "No Clear Signal",
        "action": "Monitor",
        "human_reason": "Available signals do not provide enough evidence for a specific intervention."
    }
}
playbook["archetype"] = playbook["reason_code"].map(
    lambda x: archetype_map[x]["archetype"]
)
playbook["human_reason"] = playbook["reason_code"].map(
    lambda x: archetype_map[x]["human_reason"]
)
print("Archetype mapping created successfully!")
display(
    playbook[
        [
            "action_rank",
            "archetype",
            "action",
            "reason_code",
            "human_reason"
        ]
    ].head(10)
)

Archetype mapping created successfully!


,action_rank,archetype,action,reason_code,human_reason
0,1,Position Declining,Review and Refresh,POSITION_DECLINE,Recent average position moved materially worse...
1,2,Position Declining,Review and Refresh,POSITION_DECLINE,Recent average position moved materially worse...
2,3,Position Declining,Review and Refresh,POSITION_DECLINE,Recent average position moved materially worse...
3,4,Position Declining,Review and Refresh,POSITION_DECLINE,Recent average position moved materially worse...
4,5,Position Declining,Review and Refresh,POSITION_DECLINE,Recent average position moved materially worse...
5,6,Position Declining,Review and Refresh,POSITION_DECLINE,Recent average position moved materially worse...
6,7,Position Declining,Review and Refresh,POSITION_DECLINE,Recent average position moved materially worse...
7,8,Position Declining,Review and Refresh,POSITION_DECLINE,Recent average position moved materially worse...
8,9,Position Declining,Review and Refresh,POSITION_DECLINE,Recent average position moved materially worse...
9,10,Position Declining,Review and Refresh,POSITION_DECLINE,Recent average position moved materially worse...


## 2. Intended use and limits

Intended use:
This playbook is for content and search teams, intended as a decision support for assessing which content/query pairs require human intervention. This section describes the ranked queue of content/query pairs that should be reviewed.

This ranked list can be used for prioritizing the items reviewed by humans, such as:
- content with observed drops in average position
- content with observed drops in CTR
- content with no obvious signals for intervention and therefore requiring closer inspection.

The following workflow is suggested for items added to the queue:

ranked signal -> human review -> contextual analysis -> decision -> action -> measurement

Limits:
The playbook is directional and should not be used as a production readiness system for any content or search-related decisions. The described signals are indicative but do not prove causality; the content under review is not guaranteed to benefit from the proposed changes. Moreover, the signals are based on data for a limited time period, and some content records may have fewer signals due to this limitation.

When evidence is unavailable or insufficient, it is better to not make any decisions or ask for further human review rather than act on misleading information. The thresholds and labels described in this document were derived from the underlying data and should be re-evaluated if the distribution of the data, the clients it represents, or the content under review changes substantially.

In [21]:
required_playbook_columns = [
    "action_rank",
    "client_hash_id",
    "content_hash_id",
    "query_hash_id",
    "archetype",
    "action",
    "reason_code",
    "human_reason",
    "priority_score"
]

missing_columns = [
    col for col in required_playbook_columns
    if col not in playbook.columns
]

if not missing_columns:
    print("PASS: All required playbook fields are present.")
else:
    print("Missing columns:", missing_columns)

print("\nPlaybook shape:", playbook.shape)

PASS: All required playbook fields are present.

Playbook shape: (2414248, 32)


## 3. Human review + the no-go list

Human review rules:
Every suggested change must be reviewed by a human before making any change to the content.
A reviewer should:
- Understand the context of the change to be reviewed.
- Determine that the signal observed is genuine and not the result of missing data.
- Determine if there are any recent business, technical, or content changes that might explain the observed signal.
- Evaluate the reason code and accompanying signals rather than the change itself.
- Log the decision and, if possible, measure the impact of the change.

Only the highest-priority items should be reviewed.

The system should not, without human review:
- Publish or unpublish content
- Edit content
- Delete or de-index content
- Make ranking or search configuration changes
- Make client facing decisions such as claiming that a proposed change will improve performance
- Act on items with missing or ambiguous signals

A human must review decisions when signals were weak, the proposed change has high impact, or the reviewer cannot determine if the suggested change is appropriate.

In [22]:
# Week 7 - Human review and no-go checks

review_rules = pd.DataFrame({
    "Rule": [
        "Review the underlying content/query context",
        "Check signal quality and missing data",
        "Check for external/contextual explanations",
        "Treat ranking as review priority, not automatic action",
        "Measure outcomes after approved changes"
    ],
    "Required": [True, True, True, True, True]
})

no_go_actions = pd.DataFrame({
    "Action": [
        "Automatically publish/edit content",
        "Automatically delete/suppress content",
        "Automatically change search configuration",
        "Make client-facing decisions without review",
        "Claim guaranteed improvement",
        "Infer causality from observed signals"
    ],
    "Automate": [False, False, False, False, False, False]
})

print("HUMAN REVIEW RULES")
display(review_rules)

print("\nNO-GO AUTOMATION LIST")
display(no_go_actions)


HUMAN REVIEW RULES


,Rule,Required
0,Review the underlying content/query context,True
1,Check signal quality and missing data,True
2,Check for external/contextual explanations,True
3,"Treat ranking as review priority, not automati...",True
4,Measure outcomes after approved changes,True



NO-GO AUTOMATION LIST


,Action,Automate
0,Automatically publish/edit content,False
1,Automatically delete/suppress content,False
2,Automatically change search configuration,False
3,Make client-facing decisions without review,False
4,Claim guaranteed improvement,False
5,Infer causality from observed signals,False


## 4. Monitoring / retrain triggers

Monitoring and retrain triggers:
I propose to use the following set of signals for monitoring production use of the playbook. Contrary to fully automated production systems, I suggest to use a light-weight approach here, where a change in any of these signals would trigger a manual review but not an immediate retraining.

Monitoring triggers:
Signal drift: if the distribution of CTR, position change or other signals used in the playbook differs from what was observed during development by more than some threshold, we should revisit the playbook.

Action mix drift: if the mix of Review and Refresh, Review Metadata or Monitor actions differs from what was observed during development by more than some threshold, we should revisit the playbook.

Missing-data increase: if the fraction of records for which a signal was unavailable increased compared to what was observed during development for the same time period, we should revisit the data pipeline.

Performance drift: if the performance of the playbook drops by more than some threshold on recent data, we should revisit the playbook.

Schema or data changes: if any features, their definitions or the time windows used to compute them changed, we should run the feature and leakage audits again.

New clients/mix: if the population of users or items changed compared to what was used during development, we should revisit the playbook.

In all of these cases, the playbook may need to be retrained if the signals it uses or the decision it makes have stopped reflecting the patterns in the data. However, I would emphasize that these are monitoring triggers rather than automated retraining triggers: in each case, a human should investigate if and why the change happened.

In [23]:
# Week 7 - Monitoring snapshot

monitoring_snapshot = pd.DataFrame({
    "Metric": [
        "Total records",
        "Review and Refresh share",
        "Review Metadata share",
        "Monitor share",
        "CTR 90d median",
        "Position change median"
    ],
    "Value": [
        len(playbook),
        (playbook["action"] == "Review and Refresh").mean(),
        (playbook["action"] == "Review Metadata").mean(),
        (playbook["action"] == "Monitor").mean(),
        playbook["ctr_90d"].median(),
        playbook["position_change"].median()
    ]
})

print("Monitoring snapshot:")
display(monitoring_snapshot)


Monitoring snapshot:


,Metric,Value
0,Total records,2.414248e+06
1,Review and Refresh share,7.072679e-02
2,Review Metadata share,3.111859e-02
3,Monitor share,8.981546e-01
4,CTR 90d median,0.000000e+00
5,Position change median,2.000000e-01


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [24]:
# Week 7 - Export ranked action queue for the paper
import os
os.makedirs("work/outputs", exist_ok=True)

# Export the ranked queue
queue_columns = [
    "action_rank",
    "client_hash_id",
    "content_hash_id",
    "query_hash_id",
    "archetype",
    "action",
    "reason_code",
    "human_reason",
    "priority_score",
    "ctr_90d",
    "ctr_last30",
    "ctr_prev30",
    "ctr_change",
    "position_change"
]

ranked_queue = playbook[queue_columns].copy()
output_path = "work/outputs/w07_ranked_action_queue.csv"
ranked_queue.to_csv(
    output_path,
    index=False
)

print("Ranked queue exported successfully!")
print("File:", output_path)
print("Rows:", len(ranked_queue))
print("Columns:", len(ranked_queue.columns))

Ranked queue exported successfully!
File: work/outputs/w07_ranked_action_queue.csv
Rows: 2414248
Columns: 14


In [25]:
print("File exists:", os.path.exists(output_path))

check_queue = pd.read_csv(output_path)

print("Loaded exported rows:", len(check_queue))
print("\nTop 5 exported actions:")

display(
    check_queue.head(5)
)

File exists: True
Loaded exported rows: 2414248

Top 5 exported actions:


,action_rank,client_hash_id,content_hash_id,query_hash_id,archetype,action,reason_code,human_reason,priority_score,ctr_90d,ctr_last30,ctr_prev30,ctr_change,position_change
0,1,client_23a62021009f63c4,content_884399c4f70bbedc,query_23797ac3cf508604,Position Declining,Review and Refresh,POSITION_DECLINE,Recent average position moved materially worse...,581.500000,0.000000,0.0,0.0,0.0,581.500000
1,2,client_20259bd6705d81d4,content_f90fd9a7a7ef5bc3,query_3115c08d1c6cb1f8,Position Declining,Review and Refresh,POSITION_DECLINE,Recent average position moved materially worse...,552.600000,0.052632,0.0,0.0,0.0,552.600000
2,3,client_20259bd6705d81d4,content_6cffe9e76a03d4e4,query_d52187abb4d40985,Position Declining,Review and Refresh,POSITION_DECLINE,Recent average position moved materially worse...,505.055556,0.000000,0.0,0.0,0.0,505.055556
3,4,client_20259bd6705d81d4,content_ab294a2f95286b64,query_be305f1213d9af03,Position Declining,Review and Refresh,POSITION_DECLINE,Recent average position moved materially worse...,433.000000,0.000000,0.0,0.0,0.0,433.000000
4,5,client_23a62021009f63c4,content_cd32ab23b32de847,query_f6dfa923811062c4,Position Declining,Review and Refresh,POSITION_DECLINE,Recent average position moved materially worse...,409.000000,0.000000,0.0,0.0,0.0,409.000000


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.